# Upsell Prediction with LightGBM
## Using Snowflake Phone Usage Data with Feature Engineering

This notebook trains an upsell prediction model using:
- **Data Source**: Snowflake tables (PHONE_USAGE_DATA, ACCOUNT_ATTRIBUTES_MONTHLY)
- **Model**: LightGBM Classifier
- **Features**: Usage metrics with rolling windows and difference features
- **Target**: Predict if account will increase usage (upsell opportunity)

**Upsell Definition**: An account is considered an upsell candidate if their total calls increase by ≥10% over the next 3 months.

---

## 1. Setup and Imports

In [ ]:
# Core libraries
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import pickle
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# LightGBM
import lightgbm as lgb

# Sklearn
from sklearn import metrics
from sklearn.metrics import (
    auc, roc_curve, precision_score, recall_score,
    f1_score, confusion_matrix, precision_recall_curve,
    classification_report, roc_auc_score
)

# Snowflake
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lit, count, sum as spark_sum

# Progress bar
from tqdm import tqdm

print("✓ Libraries imported successfully")
print(f"✓ LightGBM version: {lgb.__version__}")

## 2. Connect to Snowflake

In [ ]:
# Get active Snowflake session
session = get_active_session()

# Set database and schema
session.use_database("MY_DATABASE")
session.use_schema("PUBLIC")

print("✓ Snowflake session active")
print(f"  Database: {session.get_current_database()}")
print(f"  Schema: {session.get_current_schema()}")
print(f"  Warehouse: {session.get_current_warehouse()}")
print(f"  Role: {session.get_current_role()}")

# Verify tables exist
print("\n✓ Verifying tables exist...")
tables_to_check = ["PHONE_USAGE_DATA", "ACCOUNT_ATTRIBUTES_MONTHLY"]
for table in tables_to_check:
    count = session.table(f"MY_DATABASE.PUBLIC.{table}").count()
    print(f"  {table}: {count:,} rows")

## 3. Load Data from Snowflake

In [ ]:
# Load data from Snowflake
print("Loading data from MY_DATABASE.PUBLIC...")

# 1. Usage data
usage_df = session.table("MY_DATABASE.PUBLIC.PHONE_USAGE_DATA").to_pandas()
usage_df['MONTH'] = pd.to_datetime(usage_df['MONTH'])
print(f"✓ PHONE_USAGE_DATA: {len(usage_df):,} rows")

# 2. Account attributes
account_df = session.table("MY_DATABASE.PUBLIC.ACCOUNT_ATTRIBUTES_MONTHLY").to_pandas()
account_df['MONTH'] = pd.to_datetime(account_df['MONTH'])
print(f"✓ ACCOUNT_ATTRIBUTES_MONTHLY: {len(account_df):,} rows")

print(f"\nData date range: {usage_df['MONTH'].min()} to {usage_df['MONTH'].max()}")
print(f"Unique accounts: {usage_df['USERID'].nunique():,}")

# Display schema
print("\n=== PHONE_USAGE_DATA Schema ===")
print(usage_df.dtypes)

print("\n=== Sample Usage Data ===")
print(usage_df.head())

## 4. Feature Engineering

Following the approach from the migration notebook:
- Create difference features (3-month and 6-month changes)
- Create ratio features
- Create rolling aggregates

In [ ]:
# Merge usage data with account attributes
df = usage_df.merge(
    account_df[['ENTERPRISE_ACCOUNT_ID', 'MONTH', 'PACKAGE_ID', 'TIER_ID']],
    left_on=['USERID', 'MONTH'],
    right_on=['ENTERPRISE_ACCOUNT_ID', 'MONTH'],
    how='left'
).drop('ENTERPRISE_ACCOUNT_ID', axis=1)

# Sort data by account and month
df = df.sort_values(['USERID', 'MONTH']).reset_index(drop=True)

print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

# Configuration
PREDICTION_WINDOW = 3  # Predict 3 months ahead
UPSELL_THRESHOLD = 0.10  # 10% increase threshold

print(f"\nParameters:")
print(f"  Prediction window: {PREDICTION_WINDOW} months")
print(f"  Upsell threshold: {UPSELL_THRESHOLD:.0%} increase")

# ============================================================
# 1. TENURE AND ARR FEATURES
# ============================================================
print("\n📊 Creating tenure and ARR features...")

# Calculate signup date (first month seen for each account)
account_signup = df.groupby('USERID')['MONTH'].min().reset_index()
account_signup.columns = ['USERID', 'signup_date']
df = df.merge(account_signup, on='USERID', how='left')

# Calculate tenure in months
df['tenure_months'] = ((df['MONTH'] - df['signup_date']).dt.days / 30.44).round().astype(int)

# Create ARR based on package tier (simulated Annual Recurring Revenue)
# Map PACKAGE_ID to ARR ranges
package_arr_map = {
    100: 12000,   # Small Business
    200: 36000,   # Medium Business
    300: 84000,   # Large Business
    400: 180000,  # Enterprise
    500: 300000   # Enterprise Plus
}

# Assign ARR with some random variation (+/- 10%)
df['ARR'] = df['PACKAGE_ID'].map(package_arr_map).fillna(36000)  # Default to medium
df['ARR'] = df['ARR'] * np.random.uniform(0.9, 1.1, size=len(df))

# Calculate ARR changes (3-month and 6-month)
df['ARR_lag3'] = df.groupby('USERID')['ARR'].shift(3)
df['ARR_lag6'] = df.groupby('USERID')['ARR'].shift(6)
df['ARR_change_3m'] = df['ARR'] - df['ARR_lag3']
df['ARR_change_6m'] = df['ARR'] - df['ARR_lag6']
df['ARR_change_pct_3m'] = (df['ARR_change_3m'] / (df['ARR_lag3'] + 1)) * 100
df['ARR_change_pct_6m'] = (df['ARR_change_6m'] / (df['ARR_lag6'] + 1)) * 100

print(f"  ✓ Tenure and ARR features created")
print(f"    Tenure range: {df['tenure_months'].min()} - {df['tenure_months'].max()} months")
print(f"    ARR range: ${df['ARR'].min():,.0f} - ${df['ARR'].max():,.0f}")

# ============================================================
# 2. LAG FEATURES (3-month and 6-month lags)
# ============================================================
print("\n📊 Creating lag features...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'PHONE_MAU']:
    # 3-month lag
    df[f'{col_name}_lag3'] = df.groupby('USERID')[col_name].shift(3)
    # 6-month lag
    df[f'{col_name}_lag6'] = df.groupby('USERID')[col_name].shift(6)

# ============================================================
# 3. DIFFERENCE FEATURES (current - lag)
# ============================================================
print("📊 Creating difference features...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'PHONE_MAU']:
    # 3-month difference
    df[f'{col_name}_diff3'] = df[col_name] - df[f'{col_name}_lag3']
    # 6-month difference  
    df[f'{col_name}_diff6'] = df[col_name] - df[f'{col_name}_lag6']

# ============================================================
# 4. RATIO FEATURES
# ============================================================
print("📊 Creating ratio features...")

# Call type ratios
df['voice_calls_ratio'] = df['VOICE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['fax_calls_ratio'] = df['FAX_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['inbound_calls_ratio'] = df['PHONE_TOTAL_NUM_INBOUND_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['outbound_calls_ratio'] = df['PHONE_TOTAL_NUM_OUTBOUND_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)

# Device usage ratios
df['hardphone_ratio'] = df['HARDPHONE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['softphone_ratio'] = df['SOFTPHONE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['mobile_ratio'] = df['MOBILE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)

# Minutes per call
df['minutes_per_call'] = df['PHONE_TOTAL_MINUTES_OF_USE'] / (df['PHONE_TOTAL_CALLS'] + 1)

# MAU to total calls ratio
df['mau_to_calls_ratio'] = df['PHONE_MAU'] / (df['PHONE_TOTAL_CALLS'] + 1)

# ============================================================
# 5. ROLLING FEATURES (3-month and 6-month windows)
# ============================================================
print("📊 Creating rolling aggregates...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'VOICE_CALLS']:
    # 3-month rolling mean
    df[f'{col_name}_roll3_mean'] = df.groupby('USERID')[col_name].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    # 6-month rolling mean
    df[f'{col_name}_roll6_mean'] = df.groupby('USERID')[col_name].transform(
        lambda x: x.rolling(window=6, min_periods=1).mean()
    )

# ============================================================
# 6. CREATE UPSELL TARGET
# ============================================================
print("\n" + "="*70)
print("CREATING UPSELL TARGET")
print("="*70)

# Calculate current 3-month average
df['current_calls_avg'] = df.groupby('USERID')['PHONE_TOTAL_CALLS'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Get future calls (3 months ahead)
df['future_calls'] = df.groupby('USERID')['PHONE_TOTAL_CALLS'].shift(-PREDICTION_WINDOW)

# Calculate relative increase
df['calls_rel_increase'] = (
    (df['future_calls'] - df['current_calls_avg']) / 
    (df['current_calls_avg'] + 1e-6)
)

# Define upsell target
df['upsell_target'] = (
    (df['calls_rel_increase'] >= UPSELL_THRESHOLD).astype(int)
)

print(f"\n✓ Features created")
print(f"  Total features: {len(df.columns)}")
print(f"  Total rows: {len(df):,}")

# Remove rows where we can't calculate target or features
df_clean = df[
    df['future_calls'].notna() &  # Must have future data
    df['PHONE_TOTAL_CALLS_lag3'].notna()  # Must have at least 3 months history
].copy()

print(f"\n✓ After filtering:")
print(f"  Rows with valid target: {len(df_clean):,}")
print(f"  Upsell rate: {df_clean['upsell_target'].mean():.2%}")
print(f"  Upsell samples: {df_clean['upsell_target'].sum():,}")
print(f"  Non-upsell samples: {(len(df_clean) - df_clean['upsell_target'].sum()):,}")


## 5. Define Feature Sets

In [ ]:
# Define predictors (features to use in the model)
print("\n" + "="*70)
print("DEFINING FEATURE SETS")
print("="*70)

# Base usage features
base_features = [
    'PHONE_TOTAL_CALLS',
    'PHONE_TOTAL_MINUTES_OF_USE',
    'VOICE_CALLS',
    'FAX_CALLS',
    'PHONE_TOTAL_NUM_INBOUND_CALLS',
    'PHONE_TOTAL_NUM_OUTBOUND_CALLS',
    'HARDPHONE_CALLS',
    'SOFTPHONE_CALLS',
    'MOBILE_CALLS',
    'PHONE_MAU'
]

# Account features (NEW)
account_features = [
    'tenure_months',
    'ARR',
    'PACKAGE_ID',
    'TIER_ID'
]

# ARR change features (NEW)
arr_change_features = [
    'ARR_change_3m',
    'ARR_change_6m',
    'ARR_change_pct_3m',
    'ARR_change_pct_6m'
]

# Lag features
lag_features = [
    'PHONE_TOTAL_CALLS_lag3',
    'PHONE_TOTAL_CALLS_lag6',
    'PHONE_TOTAL_MINUTES_OF_USE_lag3',
    'PHONE_TOTAL_MINUTES_OF_USE_lag6',
    'PHONE_MAU_lag3',
    'PHONE_MAU_lag6'
]

# Difference features
diff_features = [
    'PHONE_TOTAL_CALLS_diff3',
    'PHONE_TOTAL_CALLS_diff6',
    'PHONE_TOTAL_MINUTES_OF_USE_diff3',
    'PHONE_TOTAL_MINUTES_OF_USE_diff6',
    'PHONE_MAU_diff3',
    'PHONE_MAU_diff6'
]

# Ratio features
ratio_features = [
    'voice_calls_ratio',
    'fax_calls_ratio',
    'inbound_calls_ratio',
    'outbound_calls_ratio',
    'hardphone_ratio',
    'softphone_ratio',
    'mobile_ratio',
    'minutes_per_call',
    'mau_to_calls_ratio'
]

# Rolling features
rolling_features = [
    'PHONE_TOTAL_CALLS_roll3_mean',
    'PHONE_TOTAL_CALLS_roll6_mean',
    'PHONE_TOTAL_MINUTES_OF_USE_roll3_mean',
    'PHONE_TOTAL_MINUTES_OF_USE_roll6_mean',
    'VOICE_CALLS_roll3_mean',
    'VOICE_CALLS_roll6_mean'
]

# Combine all features
predictors = (base_features + account_features + arr_change_features + 
              lag_features + diff_features + ratio_features + rolling_features)

print(f"\n📊 Feature Groups:")
print(f"  Base features: {len(base_features)}")
print(f"  Account features (tenure, ARR): {len(account_features)}")
print(f"  ARR change features: {len(arr_change_features)}")
print(f"  Lag features: {len(lag_features)}")
print(f"  Difference features: {len(diff_features)}")
print(f"  Ratio features: {len(ratio_features)}")
print(f"  Rolling features: {len(rolling_features)}")
print(f"  Total predictors: {len(predictors)}")

# Check for missing values in predictors
missing_counts = df_clean[predictors].isnull().sum()
if missing_counts.sum() > 0:
    print(f"\n⚠️ Features with missing values:")
    print(missing_counts[missing_counts > 0])
    print(f"\n  Filling missing values with 0...")
    df_clean[predictors] = df_clean[predictors].fillna(0)
else:
    print(f"\n✓ No missing values in predictors")


## 6. Split Data (Time-based Split)

In [ ]:
print("\n" + "="*70)
print("DATA SPLIT (Time-based)")
print("="*70)

# Sort by month to ensure chronological order
df_clean = df_clean.sort_values('MONTH').reset_index(drop=True)

# Get unique months
unique_months = sorted(df_clean['MONTH'].unique())
print(f"\n📅 Data range: {unique_months[0].strftime('%Y-%m')} to {unique_months[-1].strftime('%Y-%m')}")
print(f"   Total months: {len(unique_months)}")

# Time-based split (similar to migration notebook)
# Train: up to Nov 2024
# Test: Mar 2025 onwards
train_cutoff = pd.to_datetime('2024-11-01')
test_start = pd.to_datetime('2025-03-01')

# Create splits
train_df = df_clean[df_clean['MONTH'] < train_cutoff].copy()
test_df = df_clean[df_clean['MONTH'] >= test_start].copy()

# Validation set: last 20% of training data
val_size = int(len(train_df) * 0.2)
val_df = train_df.tail(val_size).copy()
train_df = train_df.head(len(train_df) - val_size).copy()

print(f"\n📊 Data Split:")
print(f"  Train: {len(train_df):,} samples ({len(train_df)/len(df_clean):.1%})")
print(f"    Date range: {train_df['MONTH'].min().strftime('%Y-%m')} to {train_df['MONTH'].max().strftime('%Y-%m')}")
print(f"    Upsell rate: {train_df['upsell_target'].mean():.2%}")

print(f"\n  Validation: {len(val_df):,} samples ({len(val_df)/len(df_clean):.1%})")
print(f"    Date range: {val_df['MONTH'].min().strftime('%Y-%m')} to {val_df['MONTH'].max().strftime('%Y-%m')}")
print(f"    Upsell rate: {val_df['upsell_target'].mean():.2%}")

print(f"\n  Test: {len(test_df):,} samples ({len(test_df)/len(df_clean):.1%})")
print(f"    Date range: {test_df['MONTH'].min().strftime('%Y-%m')} to {test_df['MONTH'].max().strftime('%Y-%m')}")
print(f"    Upsell rate: {test_df['upsell_target'].mean():.2%}")

# Extract X and y
X_train = train_df[predictors]
y_train = train_df['upsell_target']

X_val = val_df[predictors]
y_val = val_df['upsell_target']

X_test = test_df[predictors]
y_test = test_df['upsell_target']

print(f"\n✓ Feature matrices created")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  X_test shape: {X_test.shape}")

## 7. Train LightGBM Model

In [ ]:
print("\n" + "="*70)
print("TRAINING LIGHTGBM MODEL")
print("="*70)

# Model parameters (from migration notebook)
model_params = {
    "num_iterations": 100,
    "learning_rate": 0.1,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "metric": "auc",
    "random_state": 42,
    "verbose": -1
}

print(f"\n📋 Model Parameters:")
for param, value in model_params.items():
    print(f"  {param}: {value}")

# Initialize model
model = lgb.LGBMClassifier(**model_params)

# Train model with evaluation set
print(f"\n🚀 Training model...")

model.fit(
    X_train, 
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.log_evaluation(period=10)
    ]
)

print(f"\n✓ Model training completed!")

## 8. Feature Importance

In [ ]:
# Get feature importance
importance = model.feature_importances_
feat_importance = pd.DataFrame({
    "feature": predictors,
    "importance": importance
}).sort_values("importance", ascending=False)

print("\n" + "="*70)
print("TOP 20 FEATURE IMPORTANCES")
print("="*70)
print(feat_importance.head(20).to_string(index=False))

# Plot feature importance
top_n = 20
plt.figure(figsize=(10, 8))
plt.barh(
    feat_importance["feature"].iloc[:top_n][::-1],
    feat_importance["importance"].iloc[:top_n][::-1],
    color='steelblue'
)
plt.xlabel('Importance', fontsize=10)
plt.ylabel('Feature', fontsize=10)
plt.title(f"Top {top_n} Feature Importances", fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f"\n✓ Feature importance plotted")

## 9. Model Evaluation

In [ ]:
print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

# Predictions
y_train_pred = model.predict_proba(X_train)[:, 1]
y_val_pred = model.predict_proba(X_val)[:, 1]
y_test_pred = model.predict_proba(X_test)[:, 1]

# Calculate metrics
def evaluate_predictions(y_true, y_pred_proba, threshold=0.5, dataset_name="Dataset"):
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    auc_score = roc_auc_score(y_true, y_pred_proba)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n📊 {dataset_name} Results:")
    print(f"  AUC-ROC:   {auc_score:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    
    return auc_score, precision, recall, f1

# Evaluate all sets
train_auc, train_precision, train_recall, train_f1 = evaluate_predictions(
    y_train, y_train_pred, dataset_name="Training Set"
)

val_auc, val_precision, val_recall, val_f1 = evaluate_predictions(
    y_val, y_val_pred, dataset_name="Validation Set"
)

test_auc, test_precision, test_recall, test_f1 = evaluate_predictions(
    y_test, y_test_pred, dataset_name="Test Set"
)

In [ ]:
# Confusion Matrix for Test Set
threshold = 0.5
y_test_pred_binary = (y_test_pred >= threshold).astype(int)
cm = confusion_matrix(y_test, y_test_pred_binary, labels=[0, 1])

print("\n" + "="*70)
print("TEST SET CONFUSION MATRIX")
print("="*70)
if cm.shape == (2, 2):
    print(f"                   Predicted")
    print(f"              Non-upsell  Upsell")
    print(f"Actual Non-upsell  {cm[0,0]:6d}    {cm[0,1]:6d}")
    print(f"       Upsell      {cm[1,0]:6d}    {cm[1,1]:6d}")
else:
    print(cm)

# Classification report
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("="*70)
print(classification_report(y_test, y_test_pred_binary, 
                          target_names=['Non-upsell', 'Upsell'], 
                          zero_division=0))

## 10. Visualizations

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Train ROC
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred)
axes[0].plot(fpr_train, tpr_train, label=f'Train AUC = {train_auc:.3f}', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
axes[0].set_xlabel('False Positive Rate', fontsize=10)
axes[0].set_ylabel('True Positive Rate', fontsize=10)
axes[0].set_title('ROC Curve - Training Set', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Test ROC
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred)
axes[1].plot(fpr_test, tpr_test, label=f'Test AUC = {test_auc:.3f}', linewidth=2, color='green')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontsize=10)
axes[1].set_ylabel('True Positive Rate', fontsize=10)
axes[1].set_title('ROC Curve - Test Set', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ ROC curves plotted")

In [ ]:
# Confusion Matrix Visualization
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.tick_params(labelsize=9)

ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Non-upsell', 'Upsell'],
       yticklabels=['Non-upsell', 'Upsell'])
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')

# Add text annotations
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, f'{cm[i, j]:,}\n({cm[i, j]/cm.sum()*100:.1f}%)',
                      ha="center", va="center",
                      color="white" if cm[i, j] > thresh else "black",
                      fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Confusion matrix plotted")

In [ ]:
# Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Train PR
precision_train, recall_train, _ = precision_recall_curve(y_train, y_train_pred)
axes[0].plot(recall_train, precision_train, linewidth=2)
axes[0].set_xlabel('Recall', fontsize=10)
axes[0].set_ylabel('Precision', fontsize=10)
axes[0].set_title('Precision-Recall Curve - Training Set', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Test PR
precision_test, recall_test, _ = precision_recall_curve(y_test, y_test_pred)
axes[1].plot(recall_test, precision_test, linewidth=2, color='green')
axes[1].set_xlabel('Recall', fontsize=10)
axes[1].set_ylabel('Precision', fontsize=10)
axes[1].set_title('Precision-Recall Curve - Test Set', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Precision-Recall curves plotted")

## 11. Generate Predictions

In [ ]:
# Create predictions dataframe for test set
predictions_df = test_df[['USERID', 'MONTH']].copy()
predictions_df['upsell_probability'] = y_test_pred
predictions_df['predicted_upsell'] = (y_test_pred >= 0.5).astype(int)
predictions_df['actual_upsell'] = y_test.values

print("\n" + "="*70)
print("PREDICTIONS GENERATED")
print("="*70)
print(f"\n✓ Generated predictions for {len(predictions_df):,} test samples")
print(f"\nSample predictions:")
print(predictions_df.head(10))

# High probability upsell opportunities
print(f"\n=== High Upsell Probability (≥ 0.7) ===")
high_prob = predictions_df[predictions_df['upsell_probability'] >= 0.7].sort_values(
    'upsell_probability', ascending=False
)
print(f"Found {len(high_prob):,} high-probability upsell opportunities")
print(high_prob.head(10))

## 12. Save Results to Snowflake

In [ ]:
# Save predictions to Snowflake
print("\n" + "="*70)
print("SAVING TO SNOWFLAKE")
print("="*70)

print("\n📤 Saving predictions to MY_DATABASE.PUBLIC.UPSELL_PREDICTIONS...")
try:
    predictions_snowpark = session.create_dataframe(predictions_df)
    predictions_snowpark.write.mode("overwrite").save_as_table("MY_DATABASE.PUBLIC.UPSELL_PREDICTIONS")
    
    result_count = session.table("MY_DATABASE.PUBLIC.UPSELL_PREDICTIONS").count()
    print(f"✓ Saved {result_count:,} predictions to UPSELL_PREDICTIONS table")
except Exception as e:
    print(f"✗ Error saving predictions: {str(e)}")

In [ ]:
# Save model metrics
print("\n📤 Saving model metrics to MY_DATABASE.PUBLIC.UPSELL_MODEL_METRICS...")

try:
    metrics_df = pd.DataFrame({
        'model_name': ['LightGBM_Upsell'],
        'train_date': [datetime.now()],
        'test_auc': [test_auc],
        'test_precision': [test_precision],
        'test_recall': [test_recall],
        'test_f1_score': [test_f1],
        'num_features': [len(predictors)],
        'num_iterations': [model_params['num_iterations']],
        'learning_rate': [model_params['learning_rate']],
        'train_samples': [len(train_df)],
        'test_samples': [len(test_df)],
        'upsell_threshold': [UPSELL_THRESHOLD],
        'prediction_window_months': [PREDICTION_WINDOW]
    })
    
    metrics_snowpark = session.create_dataframe(metrics_df)
    metrics_snowpark.write.mode("append").save_as_table("MY_DATABASE.PUBLIC.UPSELL_MODEL_METRICS")
    
    print(f"✓ Saved model metrics to UPSELL_MODEL_METRICS table")
except Exception as e:
    print(f"✗ Error saving metrics: {str(e)}")

## 13. Save Model Files

In [ ]:
print("\n" + "="*70)
print("SAVING MODEL FILES")
print("="*70)

# Create model package
model_package = {
    'version': 'v1',
    'model': model,
    'predictors': predictors,
    'model_params': model_params,
    'threshold': 0.5,
    'test_metrics': {
        'auc': test_auc,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1
    },
    'metadata': {
        'version': 'v1',
        'description': 'LightGBM model for upsell prediction',
        'train_date': datetime.now().isoformat(),
        'num_features': len(predictors),
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'upsell_threshold': UPSELL_THRESHOLD,
        'prediction_window_months': PREDICTION_WINDOW
    }
}

# Save locally
with open('upsell_model_lgbm_v1.pkl', 'wb') as f:
    pickle.dump(model_package, f)
print("✓ Saved: upsell_model_lgbm_v1.pkl")

# Upload to Snowflake stage
print("\n📤 Uploading to Snowflake...")
try:
    session.sql("CREATE STAGE IF NOT EXISTS MY_DATABASE.PUBLIC.MODELS").collect()
    
    session.file.put(
        'upsell_model_lgbm_v1.pkl',
        '@MY_DATABASE.PUBLIC.MODELS/upsell/v1/',
        auto_compress=False,
        overwrite=True
    )
    print("✓ Uploaded to: @MY_DATABASE.PUBLIC.MODELS/upsell/v1/upsell_model_lgbm_v1.pkl")
    
    # Register in model registry
    session.sql("""
        CREATE TABLE IF NOT EXISTS MY_DATABASE.PUBLIC.UPSELL_MODEL_REGISTRY (
            VERSION VARCHAR(50) PRIMARY KEY,
            MODEL_PATH VARCHAR(500),
            MODEL_TYPE VARCHAR(100),
            DESCRIPTION VARCHAR(1000),
            AUC FLOAT,
            PRECISION FLOAT,
            RECALL FLOAT,
            F1_SCORE FLOAT,
            TRAIN_DATE TIMESTAMP,
            IS_PRODUCTION BOOLEAN DEFAULT FALSE,
            UPSELL_THRESHOLD FLOAT,
            PREDICTION_WINDOW_MONTHS INT
        )
    """).collect()
    
    session.sql(f"""
        MERGE INTO MY_DATABASE.PUBLIC.UPSELL_MODEL_REGISTRY AS target
        USING (SELECT 'v1' AS VERSION) AS source
        ON target.VERSION = source.VERSION
        WHEN MATCHED THEN UPDATE SET
            MODEL_PATH = '@MY_DATABASE.PUBLIC.MODELS/upsell/v1/upsell_model_lgbm_v1.pkl',
            MODEL_TYPE = 'LightGBM',
            DESCRIPTION = 'LightGBM model for upsell prediction',
            AUC = {test_auc},
            PRECISION = {test_precision},
            RECALL = {test_recall},
            F1_SCORE = {test_f1},
            TRAIN_DATE = CURRENT_TIMESTAMP(),
            UPSELL_THRESHOLD = {UPSELL_THRESHOLD},
            PREDICTION_WINDOW_MONTHS = {PREDICTION_WINDOW}
        WHEN NOT MATCHED THEN INSERT (
            VERSION, MODEL_PATH, MODEL_TYPE, DESCRIPTION, AUC, PRECISION, RECALL, F1_SCORE,
            TRAIN_DATE, UPSELL_THRESHOLD, PREDICTION_WINDOW_MONTHS
        )
        VALUES (
            'v1', '@MY_DATABASE.PUBLIC.MODELS/upsell/v1/upsell_model_lgbm_v1.pkl', 'LightGBM',
            'LightGBM model for upsell prediction',
            {test_auc}, {test_precision}, {test_recall}, {test_f1},
            CURRENT_TIMESTAMP(), {UPSELL_THRESHOLD}, {PREDICTION_WINDOW}
        )
    """).collect()
    
    print("✓ Registered in UPSELL_MODEL_REGISTRY")
    
except Exception as e:
    print(f"⚠️ Error uploading to Snowflake: {e}")

print("\n" + "="*70)
print("✓ MODEL SAVED SUCCESSFULLY")
print("="*70)
print(f"\n📊 Model: LightGBM Classifier v1")
print(f"📁 Local file: upsell_model_lgbm_v1.pkl")
print(f"☁️  Snowflake: @MY_DATABASE.PUBLIC.MODELS/upsell/v1/")
print(f"\n📈 Performance:")
print(f"  AUC: {test_auc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  F1: {test_f1:.4f}")
print(f"\n🎯 Configuration:")
print(f"  Features: {len(predictors)}")
print(f"  Upsell threshold: {UPSELL_THRESHOLD:.0%}")
print(f"  Prediction window: {PREDICTION_WINDOW} months")

## 14. Summary

This notebook successfully:
1. ✅ Loaded phone usage data from Snowflake
2. ✅ Created comprehensive features (lag, difference, ratio, rolling)
3. ✅ Defined upsell target (≥10% usage increase over 3 months)
4. ✅ Trained LightGBM classifier
5. ✅ Evaluated model performance
6. ✅ Analyzed feature importance
7. ✅ Generated predictions
8. ✅ Saved results and model to Snowflake

### Key Insights:
- Model uses gradient boosting (LightGBM) instead of deep learning
- Feature engineering focuses on temporal changes and ratios
- Time-based split ensures realistic evaluation

### Next Steps:
- Fine-tune model hyperparameters
- Experiment with different upsell thresholds
- Add more account-level features
- Deploy for production scoring